# 10 · A janela desde 2019 também ganha com 24 épocas?

Spike E4-S6 do [`docs/backlog.md`](../docs/backlog.md). As opções e a forma de decidir foram escritas **antes** de correr, no [`experiencias/e4_s6.yaml`](../experiencias/e4_s6.yaml); este notebook lê esse ficheiro (com o mesmo `ler` dos outros spikes) e não muda nada depois de ver os números. É o notebook 09 (E4-S2) com outro YAML.

**A pergunta.** O E4-S2 escolheu a janela desde 2019 com a regra de paragem por defeito do scikit-learn, para nenhuma janela partir com vantagem. O modelo em vigor treina **24 épocas fixas** (decisão 7). Aqui compara-se o modelo em vigor (desde 2022, 24 épocas, τ = 0,25) com o mesmo modelo treinado desde 2019, antes de mudar o `TRAIN_START`.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import comparison as cmp
import correr_experiencia as ce
import evaluation as ev

RAIZ = Path.cwd().parent
DATA = RAIZ / "data" / "processed"
JANELAS_DATA = RAIZ / "data" / "janelas"

EXP = ce.ler(RAIZ / "experiencias" / "e4_s6.yaml")
N_MIN, ALPHA, EPOCAS = EXP["n_min"], EXP["alpha"], EXP["epocas"]
TAU = EXP["tau"][0]
JANELAS = EXP["janelas"]  # pela ordem da comparação: da mais curta para a mais longa
MAIS_LONGA = JANELAS[-1]

val = pd.read_parquet(DATA / "validacao.parquet", columns=["idcontrato", "codigos", "visto_no_treino"])
print(f"YAML: N = {N_MIN}, alpha = {ALPHA:g}, épocas = {EPOCAS}, tau = {TAU:g}")
print("janelas:", [(j["nome"], j["inicio"]) for j in JANELAS])
print(f"validação: {len(val)} contratos")

YAML: N = 10, alpha = 1e-07, épocas = 24, tau = 0.25
janelas: [('desde_2022', '2022-01-01'), ('desde_2019', '2019-01-01')]
validação: 23945 contratos


## Os dois treinos

A janela desde 2022 **não se voltou a treinar**: é o modelo em vigor, o treino do E4-S1 com N = 10 e 24 épocas. A janela desde 2019 foi treinada no pod (`metrics/e4_s6/`), com a mesma preparação do E4-S2 (`data/janelas/2019/`). A célula confirma, no `treino.json` de cada uma, que a configuração é a do YAML e que o treino começa na data certa.

In [2]:
def nome_tau(tau):
    return "sugestoes_validacao.parquet" if tau == 0 else f"sugestoes_validacao_tau{tau:g}.parquet"


info = {j["nome"]: json.loads((RAIZ / j["pasta"] / "treino.json").read_text(encoding="utf-8")) for j in JANELAS}
for j in JANELAS:
    i = info[j["nome"]]
    if (i["n_min"], i["alpha"], i["epocas"]) != (N_MIN, ALPHA, EPOCAS):
        raise ValueError(f"{j['nome']}: o treino.json não é o que o YAML pede")
    if i["treino"]["datas"][0] < j["inicio"] or i["treino"]["datas"][1] != "2025-09-30":
        raise ValueError(f"{j['nome']}: o treino vai de {i['treino']['datas']}, não começa em {j['inicio']}")
    if i["validacao"]["contratos"] != len(val):
        raise ValueError(f"{j['nome']}: sugeriu para outra validação")
pd.DataFrame({nome: {"início": [j["inicio"] for j in JANELAS if j["nome"] == nome][0],
                     "datas do treino": " a ".join(i["treino"]["datas"]),
                     "exemplos de treino": i["exemplos_usados"], "etiquetas": i["etiquetas"],
                     "vocabulário": i["vocabulario"], "épocas até parar": max(i["n_iter"]),
                     "treinar (min)": round(i["tempo_vetorizar_treinar_s"] / 60, 1),
                     "pico RSS do treino (MiB)": i["pico_rss_mb"]}
              for nome, i in info.items()})

,desde_2022,desde_2019
início,2022-01-01,2019-01-01
datas do treino,2022-01-01 a 2025-09-30,2019-01-02 a 2025-09-30
exemplos de treino,478234,785542
etiquetas,3000,3621
vocabulário,402084,580234
épocas até parar,24,24
treinar (min),31.2,57.6
pico RSS do treino (MiB),10030,17089


## As sugestões

Um ficheiro por janela, o do τ do YAML (0,25). As verificações são as dos notebooks 04 a 08: uma linha por contrato da validação, alinhada pela mesma ordem, com 5 códigos distintos de 8 dígitos.

In [3]:
CODIGO = re.compile(r"\d{8}")


def ler_sugestoes(pasta):
    sug = pd.read_parquet(RAIZ / pasta / nome_tau(TAU), columns=["idcontrato", "sugestoes"])
    sug["idcontrato"] = sug["idcontrato"].astype(str)
    if sug["idcontrato"].duplicated().any() or set(sug["idcontrato"]) != set(val["idcontrato"].astype(str)):
        raise ValueError(f"{pasta}: os contratos não são os da validação, um por linha")
    s = sug.set_index("idcontrato")["sugestoes"].reindex(val["idcontrato"].astype(str)).map(list)
    s.index = val.index
    if not s.map(lambda l: len(set(l)) == ev.TOP_K and all(CODIGO.fullmatch(c) for c in l)).all():
        raise ValueError(f"{pasta}: sugestões mal formadas")
    return s


por = {j["nome"]: ler_sugestoes(j["pasta"]) for j in JANELAS}
pd.Series({nome: s.iloc[0] for nome, s in por.items()})

desde_2022    [45233200, 98300000, 45233220, 45233252, 45233...
desde_2019    [45233200, 45200000, 45233252, 45233220, 45233...
dtype: object

## O grupo da decisão

O YAML manda decidir nos contratos da validação cujo texto **não aparece no treino da janela mais longa** (desde 2019). É o `visto_no_treino` do `validacao.parquet` preparado com essa janela (copiado do pod para `data/janelas/2019/`).

**Porquê:** com mais anos no treino, mais textos da validação já foram vistos, e o modelo acerta neles por decorar. Usar o mesmo grupo nas duas janelas impede que a mais longa ganhe só por isso.

A célula confirma primeiro que a validação dessa janela tem exatamente os mesmos contratos e códigos que a de `data/processed/` (o E4-T4 garante-o; aqui verifica-se nos ficheiros reais).

In [4]:
ano = MAIS_LONGA["inicio"][:4]
val_longa = pd.read_parquet(JANELAS_DATA / ano / "validacao.parquet", columns=["idcontrato", "codigos", "visto_no_treino"])
if not (val_longa["idcontrato"].astype(str).tolist() == val["idcontrato"].astype(str).tolist()
        and val_longa["codigos"].tolist() == val["codigos"].tolist()):
    raise ValueError("a validação da janela mais longa não é a mesma de data/processed/")
grupo = ~val_longa["visto_no_treino"].to_numpy()
grupo = pd.Series(grupo, index=val.index)
pd.Series({"todos": len(val),
           "texto visto no treino desde 2022": int(val["visto_no_treino"].sum()),
           f"texto visto no treino desde {ano}": int(val_longa["visto_no_treino"].sum()),
           "grupo da decisão (texto novo desde " + ano + ")": int(grupo.sum())}, name="contratos")

todos                                       23945
texto visto no treino desde 2022             3706
texto visto no treino desde 2019             3795
grupo da decisão (texto novo desde 2019)    20150
Name: contratos, dtype: int64

## Os números

No grupo da decisão (o que decide) e em todos os contratos (só informação). O F1 macro mostra-se sem entrar na decisão.

In [5]:
def linha(s, mask):
    rep = ev.report(s[mask], val["codigos"][mask], val["visto_no_treino"][mask])
    return {"top-5 código": rep.loc[("todos", "codigo"), "top5"],
            "top-5 classe": rep.loc[("todos", "classe"), "top5"],
            "top-1 código": rep.loc[("todos", "codigo"), "top1"],
            "F1 macro código": rep.loc[("todos", "codigo"), "f1_macro"]}


todos = pd.Series(True, index=val.index)
quadro = pd.concat({"grupo da decisão": pd.DataFrame({n: linha(s, grupo) for n, s in por.items()}).T,
                    "todos (informação)": pd.DataFrame({n: linha(s, todos) for n, s in por.items()}).T})
quadro

top-5 código  top-5 classe  top-1 código  \
grupo da decisão   desde_2022      0.750926      0.821347      0.519937   
                   desde_2019      0.754021      0.819833      0.521818   
todos (informação) desde_2022      0.783482      0.841105      0.576560   
                   desde_2019      0.786346      0.840393      0.576158   

                               F1 macro código  
grupo da decisão   desde_2022         0.242887  
                   desde_2019         0.252248  
todos (informação) desde_2022         0.256768  
                   desde_2019         0.266521

## A decisão

A regra foi escrita no YAML antes de correr: a regra de comparação do E4-T1 (bootstrap emparelhado, 1 000 amostras; decide o top-5 no código completo se a diferença contar em pelo menos 95% das amostras, o top-5 na classe é a guarda e desempata), no grupo da decisão, pela ordem das janelas. A opção em vigor é o modelo atual (desde 2022); a desde 2019 só a substitui se ganhar. Em empate fica a de 2022.

In [6]:
em_vigor = JANELAS[0]["nome"]
passos = {}
for j in JANELAS[1:]:
    cand = j["nome"]
    duelo = cmp.compare(por[cand][grupo], por[em_vigor][grupo], val["codigos"][grupo])
    escolha = cmp.choose_from(duelo, default="b")
    for medida in ("codigo", "classe"):
        r = duelo.loc[(medida, "top5")]
        passos[(f"{cand} contra {em_vigor}", medida)] = {
            "candidata": r["a"], "em vigor": r["b"], "diferença": r["a"] - r["b"],
            "candidata melhor (amostras)": r["a_melhor"], "em vigor melhor (amostras)": r["b_melhor"],
            "conta": bool(r["conta"])}
    passos[(f"{cand} contra {em_vigor}", "fica")] = {"candidata": cand if escolha == "a" else em_vigor}
    if escolha == "a":
        em_vigor = cand
ESCOLHIDA = em_vigor
print("Janela escolhida:", ESCOLHIDA)
pd.DataFrame(passos).T

Janela escolhida: desde_2022


candidata  em vigor diferença  \
desde_2019 contra desde_2022 codigo    0.754021  0.750926  0.003095   
                             classe    0.819833  0.821347 -0.001514   
                             fica    desde_2022       NaN       NaN   

                                    candidata melhor (amostras)  \
desde_2019 contra desde_2022 codigo                       0.948   
                             classe                        0.21   
                             fica                           NaN   

                                    em vigor melhor (amostras)  conta  
desde_2019 contra desde_2022 codigo                      0.049  False  
                             classe                      0.779  False  
                             fica                          NaN    NaN

## Análise completa (informação: não muda a decisão)

A decisão acima segue a regra escrita antes de correr. Aqui mostram-se **todas** as medidas que os spikes anteriores usaram, para ver se a janela vencedora piora em alguma coisa que a regra não olha. Nada daqui decide; se algo preocupar, fica escrito como limitação.

As entidades usam o treino desde 2019 como referência para as duas janelas (como o grupo da decisão).

### 1. A avaliação completa do E4-T1

Para cada janela: código completo, classe e divisão (top-1, top-5, F1 macro) e a linha `so_divisao`, em todos os contratos e só no texto novo. Aqui "texto novo" é em relação ao treino **de cada janela** (o `visto_no_treino` do seu `validacao.parquet`), por isso o número de contratos de texto novo muda de janela para janela.

In [7]:
def validacao_da_janela(j):
    p = JANELAS_DATA / j["inicio"][:4] / "validacao.parquet"
    return p if p.exists() else DATA / "validacao.parquet"  # desde 2022: data/processed/


visto = {j["nome"]: pd.read_parquet(validacao_da_janela(j), columns=["visto_no_treino"])["visto_no_treino"] for j in JANELAS}
visto = {n: pd.Series(v.to_numpy(), index=val.index) for n, v in visto.items()}
completo = pd.concat({n: ev.report(s, val["codigos"], visto[n]) for n, s in por.items()}, axis=1)
completo.style.format(lambda x: f"{x:.4f}" if isinstance(x, float) else x)

### 2. Medidas hierárquicas

Precisão, recall e F1 hierárquicos da 1.ª sugestão (Kiritchenko), micro e macro, em todos e no texto novo de cada janela. Dão meio acerto a uma sugestão perto do código certo.

In [8]:
hier = pd.concat({n: ev.hierarchical_report(s, val["codigos"], visto[n]) for n, s in por.items()})
hier.style.format("{:.4f}")

### 3. Bootstrap completo: desde 2019 contra desde 2022

Todas as medidas × métricas (`compare`), no grupo da decisão e em todos os contratos. `a` = desde 2019, `b` = desde 2022. Uma diferença conta se um lado ganhar em pelo menos 95% das 1 000 amostras.

In [9]:
comp = pd.concat({"grupo da decisão": cmp.compare(por["desde_2019"][grupo], por["desde_2022"][grupo], val["codigos"][grupo]),
                  "todos": cmp.compare(por["desde_2019"], por["desde_2022"], val["codigos"])})
comp["quem ganha"] = comp.apply(lambda r: ("2019" if r["a"] > r["b"] else "2022") if r["conta"] else "empate", axis=1)
comp.style.format({c: "{:.4f}" for c in ["a", "b", "diferenca"]} | {c: "{:.1%}" for c in ["a_melhor", "b_melhor"]})

### 4. Sugestões genéricas e etiquetas da régua

- **1.ª sugestão ≤ grupo:** a parte das 1.ªs sugestões ao nível do grupo ou acima (nível ≤ 3, ex. `45200000`). Foi a guarda que deixou a entidade de fora no E4-S3: mais dados antigos podem levar a códigos mais genéricos.
- **1.ª sugestão na régua:** a parte das 1.ªs sugestões que são uma das 5 etiquetas mais comuns (E4-S4: um modelo mal afinado cai sempre nelas). As 5 etiquetas são as da régua de cada janela e, para comparar as duas com as mesmas etiquetas, as da janela desde 2019.
- **Nível médio da 1.ª sugestão:** quantos dígitos antes dos zeros finais (mais alto = mais específico).

O bootstrap compara desde 2019 com desde 2022 em todos os contratos.

In [10]:
from cpv_data import code_level
import baselines as bl

treino22 = pd.read_parquet(DATA / "treino.parquet", columns=["codigo", "adjudicante"])
treino19 = pd.read_parquet(JANELAS_DATA / "2019" / "treino.parquet", columns=["codigo", "adjudicante"])
top = {"desde_2022": bl.most_common_labels(treino22["codigo"], n_min=N_MIN),
       "desde_2019": bl.most_common_labels(treino19["codigo"], n_min=N_MIN)}
for n, t in top.items():
    print(f"5 etiquetas da régua {n}:", t)

nivel1 = {n: code_level(s.map(lambda l: l[0])) for n, s in por.items()}
genericas = pd.DataFrame({n: {"1.ª sugestão ≤ grupo": (nivel1[n] <= 3).mean(),
                              "nível médio da 1.ª sugestão": nivel1[n].mean(),
                              "1.ª sugestão na régua desde 2019": s.map(lambda l: l[0] in top["desde_2019"]).mean(),
                              "1.ª sugestão na régua da própria janela": s.map(lambda l: l[0] in top[n]).mean()}
                          for n, s in por.items()}).T
ind = {n: (nivel1[n] <= 3).astype(float) for n in ("desde_2019", "desde_2022")}
r = cmp.paired_bootstrap(cmp.weighted_mean(ind["desde_2019"]), cmp.weighted_mean(ind["desde_2022"]), len(val))
print(f"1.ª sugestão ≤ grupo, 2019 contra 2022: {r['a']:.2%} vs {r['b']:.2%}, "
      f"2019 maior em {r['a_melhor']:.1%} das amostras, 2022 maior em {r['b_melhor']:.1%}; conta: {r['conta']}")
genericas

5 etiquetas da régua desde_2022: ['33600000', '33140000', '33690000', '33100000', '98300000']
5 etiquetas da régua desde_2019: ['33600000', '33140000', '33690000', '33100000', '33696500']


1.ª sugestão ≤ grupo, 2019 contra 2022: 30.42% vs 30.57%, 2019 maior em 18.7% das amostras, 2022 maior em 80.8%; conta: False


,1.ª sugestão ≤ grupo,nível médio da 1.ª sugestão,1.ª sugestão na régua desde 2019,1.ª sugestão na régua da própria janela
desde_2022,0.305742,4.295636,0.284402,0.288787
desde_2019,0.304239,4.315807,0.281812,0.281812


### 5. A régua de cada janela

A régua (E4-T2) muda com a janela, porque conta as etiquetas do treino. O modelo tem de ficar muito acima dela em cada janela (estimativa pontual, sem bootstrap).

In [11]:
linhas = {}
for n in por:
    regua = bl.suggest_most_common(top[n], val.index)
    for nome_grupo, mask in (("todos", todos), ("texto novo da janela", ~visto[n])):
        for medida in ("codigo", "classe"):
            m = ev.report(por[n][mask], val["codigos"][mask], visto[n][mask]).loc[("todos", medida), "top5"]
            rg = ev.report(regua[mask], val["codigos"][mask], visto[n][mask]).loc[("todos", medida), "top5"]
            linhas[(n, nome_grupo, medida)] = {"modelo": m, "régua": rg, "acima da régua (pontos)": 100 * (m - rg)}
pd.DataFrame(linhas).T.style.format({"modelo": "{:.2%}", "régua": "{:.2%}", "acima da régua (pontos)": "{:.1f}"})

### 6. Entidades no treino e entidades novas

Os grupos do E4-S3: a entidade (`adjudicante`) de um contrato da validação "está no treino" se aparecer nas `adjudicante` do treino desde 2019 (o mesmo grupo para as duas janelas). Mostra se a janela longa ajuda sobretudo nas entidades que já conhecia.

In [12]:
def limpar(serie):
    return serie.astype("object").fillna("").str.split().str.join(" ")


adj = pd.read_parquet(DATA / "validacao.parquet", columns=["adjudicante"])["adjudicante"]
adj.index = val.index
conhecidas = limpar(adj).isin(set(limpar(treino19["adjudicante"])))
ent = {}
for nome_grupo, mask in (("entidades no treino desde 2019", conhecidas), ("entidades novas", ~conhecidas)):
    for n, s in por.items():
        rep = ev.report(s[mask], val["codigos"][mask], visto[n][mask])
        ent[(nome_grupo, n)] = {"contratos": int(mask.sum()),
                                "top-5 código": rep.loc[("todos", "codigo"), "top5"],
                                "top-5 classe": rep.loc[("todos", "classe"), "top5"]}
pd.DataFrame(ent).T.style.format({"top-5 código": "{:.2%}", "top-5 classe": "{:.2%}"})

### 7. Top-5 no código completo por nível da etiqueta

O nível da etiqueta (o 1.º código do contrato): 3 = grupo, 4 = classe, 5 = categoria, 6 a 8 = mais específico. Mostra em que tipo de código cada janela acerta mais, no grupo da decisão.

In [13]:
nivel_etq = code_level(val["codigos"].str.split().str[0])
por_nivel = {}
for nv in sorted(nivel_etq[grupo & (nivel_etq > 2)].unique()):
    mask = grupo & (nivel_etq == nv)
    por_nivel[nv] = {"contratos": int(mask.sum()),
                     **{n: ev.hits(s[mask], val["codigos"][mask], "codigo", ev.TOP_K).mean() for n, s in por.items()}}
pd.DataFrame(por_nivel).T.rename_axis("nível da etiqueta").style.format({n: "{:.2%}" for n in por})

,contratos,desde_2022,desde_2019
nível da etiqueta,,,
3,4644.000000,82.97%,83.35%
4,4963.000000,82.89%,80.80%
5,3316.000000,64.57%,66.01%
6,2360.000000,65.93%,69.07%
7,841.000000,60.76%,62.19%
8,344.000000,55.52%,56.40%


## Memória do treino desde 2019 com 24 épocas

Do `/usr/bin/time -v` do log do pod (`Maximum resident set size`, em KiB). A preparação é a do E4-S2 (secção 18).

In [14]:
def pico_kib(log):
    m = re.search(r"Maximum resident set size \(kbytes\): (\d+)", (RAIZ / log).read_text(encoding="utf-8"))
    return int(m.group(1))


memoria = {}
for j in JANELAS:
    a = j["inicio"][:4]
    for passo in ("preparar", "treino"):
        log = f"{EXP['saida']}/pod_{passo}_{a}.log"
        if (RAIZ / log).exists():
            memoria[(j["nome"], passo)] = {"pico (GB)": round(pico_kib(log) * 1024 / 1e9, 1)}
pd.DataFrame(memoria).T

,,pico (GB)
desde_2019,treino,17.9


## Conclusão

**Fica a janela desde 2022.** Com 24 épocas, a janela desde 2019 empata com o modelo em vigor, e a regra escrita no YAML antes de correr manda ficar com a de 2022 em empate. Os números são do grupo da decisão: 20 150 contratos com texto novo para o treino desde 2019, τ = 0,25.

| | desde 2022 (em vigor) | desde 2019 | diferença | desde 2019 melhor em |
|---|---|---|---|---|
| top-5 código | 75,09% | 75,40% | +0,31 | 94,8% das amostras (não conta) |
| top-5 classe | 82,13% | 81,98% | −0,15 | 21,0% (não conta) |

**Porque é que o E4-S2 deu outra coisa:**

| Top-5 código (grupo da decisão) | regra por defeito (E4-S2) | 24 épocas (aqui) |
|---|---|---|
| desde 2022 | 73,46% (parou na época 12) | 75,09% |
| desde 2019 | 75,38% (parou na época 11) | 75,40% |

Com a regra por defeito, a janela desde 2022 parava cedo demais. A desde 2019, com mais exemplos por época, chegava ao seu máximo em 11 épocas. A vantagem que o E4-S2 mediu era sobretudo de treino mais completo, não de mais dados.

**Análise completa** (informação):
- Nas 18 comparações do bootstrap, a desde 2019 ganha em 6 (F1 macro no código; top-1 e F1 macro na divisão) e empata nas outras 12.
- As sugestões genéricas ficam iguais (30,57% → 30,42%, diferença que não conta).
- Por nível da etiqueta, a desde 2019 ganha nos códigos específicos (+3,1 pontos no nível 6) e perde na classe (−2,1 pontos no nível 4).

**Custo da janela desde 2019:** 57,6 min de treino, contra 31,2, e um pico de 17,9 GB. A preparação não cabe no portátil.

**Limitação:** as 24 épocas foram afinadas na janela desde 2022. A grelha de épocas na janela desde 2019 não se correu. Decisão 1 e secção 19 do `docs/qualidade_dados.md`.